In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# QESEM: Sebuah Qiskit Function oleh Qedma
> **Note:** Qiskit Functions adalah fitur eksperimental yang hanya tersedia untuk pengguna IBM Quantum&reg; Premium Plan, Flex Plan, dan On-Prem (melalui IBM Quantum Platform API) Plan. Fitur ini dalam status rilis pratinjau dan dapat berubah sewaktu-waktu.

## Gambaran Umum
Meskipun unit pemrosesan kuantum telah berkembang pesat dalam beberapa tahun terakhir, error akibat noise dan ketidaksempurnaan di perangkat keras yang ada tetap menjadi tantangan utama bagi pengembang algoritma kuantum. Seiring bidang ini mendekati komputasi kuantum skala utilitas yang tidak dapat diverifikasi secara klasik, solusi untuk menghilangkan noise dengan akurasi terjamin menjadi semakin penting. Untuk mengatasi tantangan ini, Qedma telah mengembangkan Quantum Error Suppression and Error Mitigation (QESEM), yang terintegrasi secara mulus di IBM Quantum Platform sebagai [Qiskit Function](/guides/functions).

Dengan QESEM, pengguna bisa menjalankan sirkuit kuantum mereka di QPU yang berisik untuk mendapatkan hasil bebas error yang sangat akurat dengan overhead waktu QPU yang sangat efisien, mendekati batas fundamental. Untuk mencapai ini, QESEM memanfaatkan serangkaian metode proprietary yang dikembangkan oleh Qedma, untuk karakterisasi dan pengurangan error. Teknik pengurangan error mencakup optimisasi gate, transpilasi yang sadar noise, error suppression (ES), dan error mitigation (EM) yang tidak bias. Dengan kombinasi metode berbasis karakterisasi ini, pengguna bisa mendapatkan hasil yang andal dan bebas error untuk sirkuit kuantum bervolume besar yang generik, membuka aplikasi yang sebelumnya tidak bisa dicapai.

Untuk deskripsi lengkap komponen-komponen yang mendasarinya, serta demonstrasi skala utilitas, lihat makalah [Reliable high-accuracy error mitigation for utility-scale quantum circuits.](https://arxiv.org/abs/2508.10997)
## Deskripsi
Kamu bisa menggunakan fungsi QESEM oleh Qedma untuk dengan mudah mengestimasi dan mengeksekusi sirkuitmu dengan error suppression dan mitigation, mencapai volume sirkuit yang lebih besar dan akurasi yang lebih tinggi. Untuk menggunakan QESEM, kamu menyediakan sebuah sirkuit kuantum, sekumpulan observabel untuk diukur, target akurasi statistik untuk setiap observabel, dan QPU yang dipilih. Sebelum menjalankan sirkuit ke akurasi target, kamu bisa mengestimasi waktu QPU yang dibutuhkan berdasarkan kalkulasi analitik yang tidak memerlukan eksekusi sirkuit. Setelah puas dengan estimasi waktu QPU, kamu bisa mengeksekusi sirkuit dengan QESEM.

Saat kamu mengeksekusi sebuah sirkuit, QESEM menjalankan protokol karakterisasi perangkat yang disesuaikan dengan sirkuitmu, menghasilkan model noise yang andal untuk error yang terjadi dalam sirkuit. Berdasarkan karakterisasi tersebut, QESEM pertama-tama mengimplementasikan transpilasi yang sadar noise untuk memetakan sirkuit input ke sekumpulan qubit fisik dan gate, yang meminimalkan noise yang memengaruhi observabel target. Ini mencakup gate yang tersedia secara native (CX/CZ di perangkat IBM&reg;), serta gate tambahan yang dioptimalkan oleh QESEM, membentuk extended gate set QESEM. QESEM kemudian menjalankan sekumpulan sirkuit ES dan EM berbasis karakterisasi di QPU dan mengumpulkan hasil pengukurannya. Ini kemudian diproses secara klasik untuk memberikan nilai ekspektasi yang tidak bias dan error bar untuk setiap observabel, sesuai dengan akurasi yang diminta.

![Gambaran umum Qedma QESEM](../docs/images/guides/qedma-qesem/overview.svg)
QESEM telah terbukti memberikan hasil akurasi tinggi untuk berbagai aplikasi kuantum dan pada volume sirkuit terbesar yang dapat dicapai saat ini. QESEM menawarkan fitur-fitur berikut yang menghadap pengguna, yang didemonstrasikan di bagian benchmark di bawah:
-	**Akurasi terjamin:** QESEM menghasilkan estimasi yang tidak bias untuk nilai ekspektasi observabel. Metode EM-nya dilengkapi dengan jaminan teoretis, yang — bersama karakterisasi mutakhir Qedma — memastikan mitigasi konvergen ke output sirkuit tanpa noise hingga akurasi yang ditentukan pengguna. Berbeda dengan banyak metode EM heuristik yang rentan terhadap error sistematis atau bias, akurasi terjamin QESEM sangat penting untuk memastikan hasil yang andal pada sirkuit kuantum dan observabel yang generik.
-	**Skalabilitas ke QPU besar:** Waktu QPU QESEM bergantung pada volume sirkuit, tetapi tidak bergantung pada jumlah qubit. Qedma telah mendemonstrasikan QESEM pada perangkat kuantum terbesar yang tersedia saat ini, termasuk IBM Quantum 127-qubit Eagle dan perangkat Heron 133-qubit.
-	**Agnostik aplikasi:** QESEM telah didemonstrasikan pada berbagai aplikasi, termasuk simulasi Hamiltonian, VQE, QAOA, dan estimasi amplitudo. Pengguna bisa memasukkan sirkuit kuantum dan observabel apa pun untuk diukur, dan mendapatkan hasil bebas error yang akurat. Satu-satunya batasan ditentukan oleh spesifikasi perangkat keras dan waktu QPU yang dialokasikan, yang menentukan volume sirkuit yang dapat diakses dan akurasi output. Sebaliknya, banyak solusi pengurangan error yang spesifik aplikasi atau melibatkan heuristik yang tidak terkontrol, sehingga tidak berlaku untuk sirkuit kuantum dan aplikasi yang generik.
-  **Extended gate set:** QESEM mendukung gate dengan sudut fraksional, dan menyediakan gate $Rzz(\theta)$ sudut fraksional yang dioptimalkan Qedma di perangkat IBM Quantum Eagle. Extended gate set ini memungkinkan kompilasi yang lebih efisien dan membuka volume sirkuit yang lebih besar hingga faktor 2 dibandingkan kompilasi CX/CZ default.
-	**Observabel multibase:** QESEM mendukung observabel input yang terdiri dari banyak Pauli string yang tidak komuter, seperti Hamiltonian generik. Pemilihan basis pengukuran dan optimisasi alokasi sumber daya QPU (shots dan sirkuit) kemudian dilakukan secara otomatis oleh QESEM untuk meminimalkan waktu QPU yang dibutuhkan untuk akurasi yang diminta. Optimisasi ini, yang memperhitungkan fidelitas perangkat keras dan laju eksekusi, memungkinkan kamu menjalankan sirkuit yang lebih dalam dan mendapatkan akurasi yang lebih tinggi.
## Benchmark
QESEM telah diuji pada berbagai kasus penggunaan dan aplikasi. Contoh-contoh berikut dapat membantu kamu menilai jenis workload apa yang bisa kamu jalankan dengan QESEM.

Ukuran kinerja utama untuk mengkuantifikasi tingkat kesulitan baik error mitigation maupun simulasi klasik untuk sirkuit dan observabel tertentu adalah **volume aktif**: jumlah gate CNOT yang memengaruhi observabel dalam sirkuit. Volume aktif bergantung pada kedalaman dan lebar sirkuit, bobot observabel, dan struktur sirkuit, yang menentukan light cone dari observabel. Untuk detail lebih lanjut, lihat pembicaraan dari [IBM Quantum Summit 2024](https://www.youtube.com/watch?v=Hd-IGvuARfE&t=1730s). QESEM memberikan nilai yang sangat besar di regime volume tinggi, memberikan hasil yang andal untuk sirkuit dan observabel yang generik.

![Volume aktif](../docs/images/guides/qedma-qesem/active_volume.svg)

| Aplikasi | Jumlah qubit | Perangkat | Deskripsi sirkuit | Akurasi | Total waktu | Penggunaan runtime |
| ---------  | ---------------- | ----- | -------------------------- | -------- | ---------- | ------------- |
| Sirkuit VQE  | 8              | Eagle (r3) | 21 total layer, 9 basis pengukuran, rantai 1D                    | 98%      | 35 menit      | 14 menit         |
| Kicked Ising   | 28              | Eagle (r3) | 3 layer unik x 3 langkah, topologi heavy-hex 2D                      | 97%     | 22 menit      | 4 menit          |
| Kicked Ising   | 28              | Eagle (r3) | 3 layer unik x 8 langkah, topologi heavy-hex 2D                     | 97%      | 116 menit      | 23 menit          |
| Simulasi Hamiltonian Trotterisasi   | 40  | Eagle (r3)            | 2 layer unik x 10 langkah Trotter, rantai 1D                    | 97%      | 3 jam     | 25 menit         |
| Simulasi Hamiltonian Trotterisasi   | 119  | Eagle (r3)           | 3 layer unik x 9 langkah Trotter, topologi heavy-hex 2D                    | 95%      | 6,5 jam     | 45 menit         |
| Kicked Ising  | 136             | Heron (r2) | 3 layer unik x 15 langkah, topologi heavy-hex 2D                    | 99%      | 52 menit             | 9 menit           |

Akurasi diukur di sini relatif terhadap nilai ideal observabel: $\frac{\langle O \rangle_{ideal} - \epsilon}{\langle O \rangle_{ideal}}$, di mana '$\epsilon$' adalah presisi absolut mitigasi (ditetapkan oleh input pengguna), dan $\langle O \rangle_{ideal}$ adalah observabel pada sirkuit tanpa noise.
'Penggunaan runtime' mengukur penggunaan benchmark dalam mode batch (jumlah penggunaan job individual), sedangkan 'total waktu' mengukur penggunaan dalam mode sesi (waktu dinding eksperimen), yang mencakup waktu klasik dan komunikasi tambahan. QESEM tersedia untuk eksekusi dalam kedua mode, sehingga pengguna bisa memanfaatkan sumber daya yang tersedia sebaik mungkin.

Sirkuit Kicked Ising 28-qubit mensimulasikan Discrete Time Quasicrystal yang dipelajari oleh Shinjo et al. (lihat [arXiv 2403.16718](https://arxiv.org/abs/2403.16718) dan [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)) pada tiga loop terhubung dari ibm_kawasaki. Parameter sirkuit yang digunakan di sini adalah $(\theta_x, \theta_z) = (0.9 \pi, 0)$, dengan keadaan awal feromagnetik $| \psi_0 \rangle = | 0 \rangle ^{\otimes n}$. Observabel yang diukur adalah nilai absolut magnetisasi $M = |\frac{1}{28} \sum_{i=0}^{27} \langle Z_i \rangle|$. Eksperimen Kicked Ising skala utilitas dijalankan pada 136 qubit terbaik dari ibm_fez; benchmark khusus ini dijalankan pada sudut Clifford $(\theta_x, \theta_z) = (\pi, 0)$, di mana volume aktif tumbuh perlahan seiring kedalaman sirkuit, yang — bersama fidelitas perangkat yang tinggi — memungkinkan akurasi tinggi pada waktu runtime yang singkat.

Sirkuit simulasi Hamiltonian Trotterisasi untuk model Ising Transverse-Field pada sudut fraksional: $(\theta_{zz}, \theta_x) = (\pi / 4, \pi /8)$ dan $(\theta_{zz}, \theta_x) = (\pi / 6, \pi / 8)$ secara berurutan (lihat [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)). Sirkuit skala utilitas dijalankan pada 119 qubit terbaik dari ibm_brisbane, sedangkan eksperimen 40-qubit dijalankan pada rantai terbaik yang tersedia. Akurasi dilaporkan untuk magnetisasi; hasil akurasi tinggi juga diperoleh untuk observabel dengan bobot yang lebih tinggi.

Sirkuit VQE dikembangkan bersama peneliti dari Center for Quantum Technology and Applications di Deutsches Elektronen-Synchrotron (DESY). Observabel target di sini adalah Hamiltonian yang terdiri dari sejumlah besar Pauli string yang tidak komuter, menekankan kinerja QESEM yang dioptimalkan untuk observabel multi-basis. Mitigasi diterapkan pada ansatz yang dioptimalkan secara klasik; meskipun hasil ini masih belum dipublikasikan, hasil dengan kualitas yang sama akan diperoleh untuk sirkuit yang berbeda dengan properti struktural yang serupa.
## Memulai
Autentikasi menggunakan [kunci API IBM Quantum Platform](http://quantum.cloud.ibm.com/)-mu, dan pilih Qiskit Function QESEM sebagai berikut. (Cuplikan ini mengasumsikan kamu sudah [menyimpan akunmu](/guides/functions#install-qiskit-functions-catalog-client) ke lingkungan lokalmu.)

In [1]:
import qiskit

from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

qesem_function = catalog.load("qedma/qesem")

## Contoh
Untuk memulai, coba contoh dasar ini tentang mengestimasi waktu QPU yang dibutuhkan untuk menjalankan QESEM untuk `pub` tertentu:

In [ ]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [ ]:
circ = qiskit.QuantumCircuit(5)
circ.cx(0, 1)
circ.cx(2, 3)
circ.cx(1, 2)
circ.cx(3, 4)

avg_magnetization = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("Z", [q], 1 / 5) for q in range(5)], num_qubits=5
)
other_observable = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("ZZ", [0, 1], 1.0), ("XZ", [1, 4], 0.5)], num_qubits=5
)

time_estimation_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    options={
        "estimate_time_only": "analytical",
    },
    backend_name=backend_name,  # E.g. "ibm_fez"
)

time_estimate_result = time_estimation_job.result()

Contoh berikut mengeksekusi sebuah job QESEM:

In [ ]:
sample_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    backend_name=backend_name,  # E.g. "ibm_fez"
)

Kamu bisa menggunakan API Qiskit Serverless yang familiar untuk memeriksa status workload Qiskit Function-mu atau mengembalikan hasilnya:

In [ ]:
print(sample_job.status())
result = sample_job.result()

## Parameter fungsi
| Nama |  Tipe | Deskripsi | Wajib | Default |  Contoh |
| -----| ------| ------------| -------- | ------- | -------- |
| `pubs` | [EstimatorPubLike](/guides/primitive-input-output) | Ini adalah input utama. `Pub` berisi 2-4 elemen: sebuah sirkuit, satu atau lebih observabel, 0 atau satu set nilai parameter, dan presisi opsional. Jika presisi tidak ditentukan, maka `default_precision` dari `options` akan digunakan |  Ya|  N/A |  `[(circuit, [obs1,obs2,obs3], parameter_values, 0.03)]`  |
| `backend_name`| string| Nama backend yang digunakan |Tidak | QESEM akan mendapatkan perangkat paling tidak sibuk yang dilaporkan oleh IBM| `"ibm_fez"`|
| `instance` | string|  Nama sumber daya cloud dari instance yang digunakan dalam format tersebut |  Tidak |  N/A | `"CRN"`  |
| `options` | dictionary | Opsi input. Lihat bagian **Opsi** untuk detail lebih lanjut. | Tidak |  Lihat bagian **Opsi** untuk detail.    |  `{ default_precision = 0.03, "max_execution_time" = 3600, "transpilation_level" = 0}`  |

### Opsi
| Opsi |  Pilihan | Deskripsi | Default |
| -----| -----------| -------- | ------- |
| `estimate_time_only` |  `"analytical"`  / `"empirical"` / None  | Saat diset, job QESEM hanya akan menghitung estimasi waktu QPU. Lihat deskripsi berikut untuk detail lebih lanjut. Saat diset ke None, sirkuit akan dieksekusi dengan QESEM | None |
|`default_precision` | 0 < float | Akan diterapkan ke `pubs` yang tidak memiliki presisi. Presisi menandakan error yang dapat diterima pada nilai ekspektasi observabel dalam nilai absolut. Yaitu, waktu runtime QPU untuk mitigasi akan ditentukan untuk memberikan nilai output untuk semua observabel yang diminati yang jatuh dalam interval kepercayaan `1σ` dari presisi target. Jika beberapa observabel disediakan, mitigasi akan berjalan hingga presisi target tercapai untuk setiap observabel input. | 0.02|
|`max_execution_time` | 0 < integer < 28.800 (8 jam)| Memungkinkan kamu membatasi waktu QPU, ditentukan dalam detik, yang digunakan untuk seluruh proses QESEM. Lihat detail tambahan di bawah.| 3.600 (satu jam)|
| `transpilation_level` | 0 / 1 | Lihat deskripsi di bawah | 1|
| `execution_mode` | `"session"` / `"batch"` | Lihat deskripsi berikut | "batch"|

> **Caution:** Estimasi waktu QPU berubah dari satu backend ke backend lain. Oleh karena itu, saat mengeksekusi fungsi QESEM, pastikan untuk menjalankannya di backend yang sama yang dipilih saat mendapatkan estimasi waktu QPU.

> **Note:** QESEM akan mengakhiri jalannya saat mencapai presisi target atau saat mencapai `max_execution_time`, mana pun yang lebih dulu.

- `estimate_time_only` - Flag ini memungkinkan pengguna mendapatkan estimasi untuk waktu QPU yang dibutuhkan untuk mengeksekusi sirkuit dengan QESEM.
    - Jika diset ke `"analytical"`, batas atas waktu QPU dihitung tanpa mengonsumsi penggunaan QPU apa pun. Estimasi ini memiliki resolusi 30 menit (misalnya, 30 menit, 60 menit, 90 menit, dan seterusnya). Estimasi ini biasanya pesimistis, dan hanya bisa diperoleh untuk observabel Pauli tunggal atau jumlah Pauli tanpa support yang berpotongan (misalnya, Z0+Z1). Ini terutama berguna untuk membandingkan tingkat kompleksitas parameter berbeda yang disediakan oleh pengguna (sirkuit, akurasi, dan sebagainya).
    - Untuk mendapatkan estimasi waktu QPU yang lebih akurat, set flag ini ke `"empirical"`. Meskipun opsi ini mengharuskan menjalankan sejumlah kecil sirkuit, opsi ini memberikan estimasi waktu QPU yang jauh lebih akurat. Estimasi ini memiliki resolusi 5 menit (misalnya, 20 menit, 25 menit, 30 menit, dan seterusnya). Pengguna dapat memilih untuk menjalankan estimasi waktu empiris dalam mode batch atau sesi. Untuk detail lebih lanjut, lihat deskripsi `execution_mode`. Misalnya, dalam mode batch, estimasi waktu empiris akan mengonsumsi kurang dari 10 menit waktu QPU.

- `max_execution_time`: Memungkinkan kamu membatasi waktu QPU, ditentukan dalam detik, yang digunakan untuk seluruh proses QESEM. Karena waktu QPU akhir yang dibutuhkan untuk mencapai akurasi target ditentukan secara dinamis selama job QESEM, parameter ini memungkinkan kamu membatasi biaya eksperimen. Jika waktu QPU yang ditentukan secara dinamis lebih pendek dari waktu yang dialokasikan oleh pengguna, parameter ini tidak akan memengaruhi eksperimen. Parameter `max_execution_time` sangat berguna dalam kasus di mana estimasi waktu analitik yang diberikan QESEM sebelum job dimulai terlalu pesimistis dan pengguna ingin memulai job mitigasi. Setelah batas waktu tercapai, QESEM berhenti mengirim sirkuit baru. Sirkuit yang sudah terkirim terus berjalan (sehingga total waktu mungkin melebihi batas hingga 30 menit), dan pengguna menerima hasil yang diproses dari sirkuit yang berjalan hingga titik tersebut. Jika kamu ingin menerapkan batas waktu QPU yang lebih pendek dari estimasi waktu analitik, konsultasikan dengan Qedma untuk mendapatkan estimasi akurasi yang dapat dicapai dalam batas waktu tersebut.

- `transpilation_level`: Setelah sirkuit dikirimkan ke QESEM, sirkuit secara otomatis menyiapkan beberapa transpilasi sirkuit alternatif dan memilih yang meminimalkan waktu QPU. Misalnya, transpilasi alternatif mungkin menggunakan gate RZZ fraksional yang dioptimalkan Qedma untuk mengurangi kedalaman sirkuit. Tentu saja, semua transpilasi setara dengan sirkuit input, dalam hal output idealnya. Untuk memberikan kontrol lebih atas transpilasi sirkuit, set transpilation level di `options`. Sementara `"transpilation_level": 1` sesuai dengan perilaku default yang dijelaskan di atas, `"transpilation_level": 0` hanya mencakup modifikasi minimal yang diperlukan untuk sirkuit asli; misalnya, 'layerification' - pengorganisasian operasi sirkuit ke dalam 'layer' gate dua-qubit simultan. Perhatikan bahwa pemetaan perangkat keras otomatis ke qubit dengan fidelitas tinggi diterapkan dalam kasus apa pun.

| transpilation_level | Deskripsi |
|:-:|:--|
| `1` | Transpilasi QESEM default. Menyiapkan beberapa transpilasi alternatif dan memilih yang meminimalkan waktu QPU. Barrier mungkin dimodifikasi dalam langkah layerification. |
| `0` | Transpilasi minimal: sirkuit yang dimitigasi akan sangat menyerupai sirkuit input secara struktural. Sirkuit yang disediakan dalam level 0 harus sesuai dengan konektivitas perangkat dan harus ditentukan dalam hal gate berikut: CX, Rzz(α), dan gate single-qubit standar (U, x, sx, rz, dan sebagainya). Barrier akan dihormati dalam langkah layerification. |

- `execution_mode` - Pengguna dapat memilih untuk menjalankan job QESEM dalam [sesi IBM](/guides/execution-modes#session-mode) yang didedikasikan atau di beberapa [batch IBM](/guides/execution-modes#batch-mode):
    -   **Mode Sesi**: Sesi lebih mahal tetapi menghasilkan waktu-ke-hasil yang lebih singkat. Setelah sesi dimulai, QPU direserved secara eksklusif untuk job QESEM. Kalkulasi penggunaan mencakup waktu yang dihabiskan untuk eksekusi QPU dan komputasi klasik terkait (dilakukan oleh QESEM dan IBM). Qiskit Function QESEM menangani pembuatan dan penutupan sesi secara otomatis. Untuk pengguna dengan akses tak terbatas ke QPU (misalnya, setup on-premises), direkomendasikan menggunakan mode sesi untuk eksekusi QESEM yang lebih cepat.
    -   **Mode Batch**: Dalam mode batch, QPU dilepaskan selama komputasi klasik, menghasilkan penggunaan QPU yang lebih rendah. Karena job batch biasanya berlangsung lebih lama, ada risiko drift perangkat keras yang lebih besar; QESEM menggabungkan langkah-langkah untuk mendeteksi dan mengompensasi drift, mempertahankan keandalan selama proses yang panjang.

> **Note:** Operasi barrier biasanya digunakan untuk menentukan layer gate dua-qubit dalam sirkuit kuantum. Dalam level 0, QESEM mempertahankan layer yang ditentukan oleh barrier. Dalam level 1, layer yang ditentukan oleh barrier dianggap sebagai satu alternatif transpilasi saat meminimalkan waktu QPU.
### Output
Output dari Circuit function adalah [PrimitiveResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult), yang berisi dua bidang:

- Satu objek [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult). Bisa diindeks langsung dari `PrimitiveResult`.

- Metadata level job.

Setiap `PubResult` berisi bidang `data` dan `metadata`.

- Bidang `data` berisi setidaknya array nilai ekspektasi (`PubResult.data.evs`) dan array error standar (`PubResult.data.stds`). Bisa juga berisi data lebih banyak, tergantung opsi yang digunakan.

- Bidang `metadata` berisi metadata level PUB (`PubResult.metadata`).

Cuplikan kode berikut menjelaskan cara mengambil estimasi waktu QPU (saat `estimate_time_only` diset):

In [ ]:
print(
    f"The estimated QPU time for this PUB is: \n{time_estimate_result[0].metadata}"
)

Cuplikan kode berikut mendemonstrasikan cara mengambil hasil mitigasi (saat `estimate_time_only` tidak diset) dan metrik eksekusi. Ini berisi data penting yang memungkinkan pemahaman lebih mendalam tentang bagaimana parameter berbeda memengaruhi eksekusi QESEM. Ini juga mungkin relevan saat menulis makalah berdasarkan penelitianmu.

In [ ]:
results = result[0]
print(f"Mitigated expectation values: \n{results.data.evs}")
print(f"Mitigated error-bar: \n{results.data.stds}")
noisy_results = results.metadata["noisy_results"]
print(f"Noisy expectation values: \n{noisy_results.evs}")
print(f"Noisy error-bar: \n{noisy_results.stds}")
print(f"Total QPU time: \n {results.metadata['total_qpu_time']}")
print(
    f"Gates fidelity measured during the experiment: \n {results.metadata['gate_fidelities']}"
)
print(
    f"Total shots / mitigation shots: \n {results.metadata['total_shots']} / {results.metadata['mitigation_shots']}"
)
print("Transpiled circuits:")
for i, circuit in enumerate(results.metadata["transpiled_circs"]):
    print(f"Circuit {i}:")
    print(f"  Circuit: \n {circuit['circuit']}")
    print(f"  Qubit mapping: \n {circuit['qubit_map']}")
    print(f"  Measurement bases: \n {circuit['num_measurement_bases']}")

## Ambil pesan error
Jika status workloadmu ERROR, gunakan job.result() untuk mengambil pesan error sebagai berikut:

In [ ]:
print(sample_job.result())

PrimitiveResult([PubResult(data=DataBin(), metadata={'time_estimation_sec': 12600})], metadata={})


## Get support

The Qedma support team is here to help! If you encounter any issues or have questions about using the QESEM Qiskit Function, please don't hesitate to reach out. Our knowledgeable and friendly support staff are ready to assist you with any technical concerns or inquiries you may have.

You can email us at support@qedma.com for assistance. Please include as much detail as possible about the issue you're experiencing to help us provide a swift and accurate response. You can also contact your dedicated Qedma POC representative via email or phone.

To help us assist you more efficiently, please provide the following information when you contact us:

- A detailed description of the issue
- The job ID
- Any relevant error messages or codes


We are committed to providing you with prompt and effective support to ensure you have the best possible experience with our Qiskit Function.

We are always looking to improve our product and we welcome your suggestions! If you have ideas on how we can enhance our services or features you'd like to see, please send us your thoughts at support@qedma.com

## Next steps

<Admonition type="tip" title="Recommendations">

- [Request access to Qedma QESEM](https://quantum.cloud.ibm.com/functions?id=qedma-qesem)
- Review [Bauman, N. P., et al. (2025). Coupled Cluster Downfolding Theory in Simulations of Chemical Systems on Quantum Hardware. arXiv preprint arXiv:2507.01199.](https://arxiv.org/pdf/2507.01199)
- Review [Aharonov, D., et al. (2025). Reliable high-accuracy error mitigation for utility-scale quantum circuits. arXiv preprint arXiv:2508.10997.](https://arxiv.org/pdf/2508.10997)


</Admonition>